# Phase 0A — vLLM serving smoke test (Kaggle T4)

**Trước khi chạy:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.

Notebook này kiểm tra Qwen3-VL có chạy được bằng vLLM trên T4 không. Thứ tự thử: Qwen3-VL-2B → Qwen3-VL-2B `--enforce-eager` → Qwen2.5-VL-3B (dự phòng).
Đừng cài Unsloth trong cùng session (xem notebook 00b).

In [ ]:
GITHUB_USER = "Licht1906"   # đổi nếu repo nằm ở tài khoản khác
REPO = "vietnamese-receipt-vlm"

import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=True)
os.chdir(f"/kaggle/working/{REPO}")
!git log --oneline -1

In [ ]:
# Cài vLLM (thay torch của Kaggle, mất vài phút) + package của repo, không kéo thêm dependency
!pip install -q "vllm>=0.11.0"
!pip install -q -e . --no-deps

In [ ]:
!python scripts/check_env.py

In [ ]:
!bash scripts/run_phase0_vllm.sh

In [ ]:
# Tùy chọn: thử thêm bản 4B (fp16 ~9 GB, vừa T4 16 GB)
# !PRIMARY=Qwen/Qwen3-VL-4B-Instruct bash scripts/run_phase0_vllm.sh

In [ ]:
SESSION = "vllm"

# Đóng gói kết quả để tải về (Output tab → /kaggle/working/phase0_results_*.zip)
import shutil, time, json, glob
for path in sorted(glob.glob("results/phase0/*.json")):
    data = json.load(open(path, encoding="utf-8"))
    print(f"{path}: status={data.get('status')}  error={data.get('error')}")
shutil.make_archive(f"/kaggle/working/phase0_results_{SESSION}_{time.strftime('%Y%m%d_%H%M')}", "zip", "results/phase0")